# D11 — E5 cố định: r=128, d=512, Position-aware Attention

Chạy toàn bộ 8 category, không chia TV1/TV2 và không cần D10 lock. Đây là **cấu hình do người dùng chọn**, chưa có bằng chứng là tối ưu. Dùng nguyên Model/Adapter/Decoder, training loop và metric hiện có.

Pipeline: Local **512** + Context đồng tâm **768→512** → frozen DINOv3 **ViT-B/16**, blocks **4/8/12** → ba Adapter r128/d512 dùng chung giữa hai view → Context→Local → sáu projection **64 kênh** → Attention `[B,6,H,W]` → logits512 → sigmoid → native Hann stitching.

Illumination là augmentation nhẹ **chỉ TRAIN**, giữ nguyên mask và áp dụng một lần trên ảnh native. Synthetic DEV giữ nguyên để chọn `best.pt` (AU-PRO@0.05; hòa giữ epoch sớm nhất). Chỉ sau khi train xong category mới evaluate toàn bộ **TEST_PUBLIC thật**, không dùng TEST để chọn checkpoint/ngưỡng. Không bảo đảm trước độ chính xác.

## 1. Đường dẫn và budget — sửa trước Run All

Chọn GPU bất kỳ trong Runtime → Change runtime type. `auto` dùng BF16 khi GPU hỗ trợ, còn lại FP16 (T4). Không cài đè CUDA PyTorch của Colab. Đẩy các file mới lên `g2/integration-D11` trước khi clone. Thay đường dẫn Drive bên dưới; đặt `RUN_NAME` mới nếu đổi protocol. Resume mặc định pin lại đúng hai Git SHA đã lưu.

Bản sửa CUDA AMP dùng namespace `illum_v2` và `outputs_v2`; không resume checkpoint/commit `e0c8881` của lần chạy lỗi v1. Chạy lại **Tham số → Mount/checkout → Chuẩn bị/preflight → Smoke GPU**; chỉ chạy full sau khi Smoke PASS. Nếu dùng notebook cũ, sửa đúng hai biến `RUN_NAME` và `LOCAL_OUTPUT` tương ứng.


In [ ]:
# @title Tham số chạy
from pathlib import Path
import json, os, shutil, subprocess, sys

GITHUB_URL = 'https://github.com/haduckien-raccoon/msila.git'
GITHUB_REF = 'g2/integration-D11'
GIT_COMMIT = ''  # Trống: resolve branch một lần; resume lấy SHA đã lưu trên Drive.
DINO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINO_COMMIT = 'main'  # Resolve một lần và lưu SHA; resume không dùng main mới.
DRIVE_PROJECT = Path('/content/drive/MyDrive/MSILA')  # SỬA
ARCHIVE_DIR = DRIVE_PROJECT / 'data'                 # 8 file <category>.tar.gz
CHECKPOINT_ON_DRIVE = DRIVE_PROJECT / 'weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'
CATEGORIES = ['can','fabric','fruit_jelly','rice','sheet_metal','vial','wallplugs','walnuts']
ARCHIVE_NAMES = {c: c + '.tar.gz' for c in CATEGORIES}
RUN_NAME = 'e5_r128_d512_attention_illum_v2'
SEED, DEV_SEED = 42, 17017
EPOCHS, BATCH_SIZE, INFERENCE_BATCH_SIZE, DATA_WORKERS = 20, 4, 4, 2
PRECISION = 'auto'  # auto | fp32 | fp16 | bf16. Giữ nguyên khi resume.
DETERMINISTIC = False  # Standard GPU bilinear kernels; geometry vẫn align_corners=False.
MAX_DEV_SOURCES = 4  # Cùng default runner G2: 4 good sources × (6 synthetic + 1 normal).
ILLUMINATION = dict(enabled=True, probability=0.5, brightness=[0.9,1.1], contrast=[0.9,1.1], gradient=0.08)
RUN_SMOKE = True    # 1 category, tối đa 2 updates; không phải kết quả nghiệm thu.
RESUME = True
WORK = Path('/content/msila_fixed')
REPO, DINO = Path('/content/msila'), Path('/content/dinov3')
DATA, ARCHIVES = WORK/'data', WORK/'archives'
WEIGHTS = WORK/'weights/dinov3_vitb16.pth'
LOCAL_OUTPUT, DRIVE_OUTPUT = WORK/'outputs_v2', DRIVE_PROJECT/'outputs/G2/E5_FIXED'/RUN_NAME
ENVIRONMENT_FILE = DRIVE_OUTPUT/'notebook_environment.json'
CONFIG = WORK/'config/fixed.yaml'
MODEL_CONFIG = WORK/'config/model.yaml'
VENV = WORK/'venv'

In [ ]:
# @title Mount Drive, pin source và dependencies
from google.colab import drive
drive.mount('/content/drive')
for directory in (WORK, DRIVE_OUTPUT, CONFIG.parent, WORK/'tmp', WORK/'mpl'):
    directory.mkdir(parents=True, exist_ok=True)
prior = json.loads(ENVIRONMENT_FILE.read_text()) if RESUME and ENVIRONMENT_FILE.is_file() else None

def checkout_pinned(url, directory, ref):
    if not (directory/'.git').is_dir():
        subprocess.run(['git','clone','--depth','1',url,str(directory)], check=True)
    dirty = subprocess.check_output(['git','-C',str(directory),'status','--porcelain','--untracked-files=no'], text=True)
    if dirty.strip(): raise RuntimeError(f'Tracked changes chưa lưu: {directory}')
    subprocess.run(['git','-C',str(directory),'fetch','--depth','1','origin',ref], check=True)
    sha = subprocess.check_output(['git','-C',str(directory),'rev-parse','FETCH_HEAD'], text=True).strip()
    subprocess.run(['git','-C',str(directory),'checkout','--detach',sha], check=True)
    return sha

MSILA_SHA = checkout_pinned(GITHUB_URL, REPO, prior['msila_commit'] if prior else GIT_COMMIT or GITHUB_REF)
DINO_SHA = checkout_pinned(DINO_URL, DINO, prior['dino_commit'] if prior else DINO_COMMIT)
if not (REPO/'scripts/run_g2_fixed.py').is_file():
    raise RuntimeError('Branch/commit chưa có notebook fixed-E5. Push code mới rồi dùng commit đó.')
if not (VENV/'bin/python').is_file():
    subprocess.run([sys.executable,'-m','venv','--system-site-packages','--without-pip',str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
requirements = [line for line in (REPO/'requirements.txt').read_text().splitlines()
                if line.strip() and not line.lstrip().startswith(('#','torch'))]
deps = WORK/'requirements_colab.txt'; deps.write_text('\n'.join(requirements)+'\n')
subprocess.run([PYTHON,'-m','pip','install','-q','-r',str(deps)], check=True)
RUN_ENV = dict(os.environ, PYTHONUNBUFFERED='1', MPLCONFIGDIR=str(WORK/'mpl'),
               TMPDIR=str(WORK/'tmp'), CUBLAS_WORKSPACE_CONFIG=':4096:8')
gpu = json.loads(subprocess.check_output([PYTHON,'-c',
    "import json,torch; assert torch.cuda.is_available(), 'GPU NOT RUN: chọn GPU Runtime'; print(json.dumps(dict(torch=str(torch.__version__),cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory,bf16_supported=torch.cuda.is_bf16_supported())))"], env=RUN_ENV, text=True))
print(dict(msila_commit=MSILA_SHA, dino_commit=DINO_SHA, seed=SEED, **gpu))

## 2. Copy tám archives xuống Colab, sau đó giải nén

Helper hiện có được mở rộng tùy chọn `include_test_public=True`: chỉ lấy TRAIN/good, VALIDATION/good và TEST_PUBLIC kèm GT; không đọc TEST_PRIVATE. Dừng nếu thiếu dữ liệu, checkpoint hoặc GT cho ảnh abnormal. Source TRAIN/DEV phải độc lập. Checkpoint pretrained được copy local và xác minh SHA-256; tuyệt đối không dùng random weights.

AMP và thứ tự tile theo ảnh giảm tải GPU/CPU; decoder/alignment dùng kernel GPU thông thường khi `DETERMINISTIC=False`. Không tuyên bố tốc độ trước khi đo trên GPU thật. Nếu OOM, tạo `RUN_NAME` mới và giảm batch xuống 2 trước khi train; không đổi cấu hình của run đã có.

In [ ]:
# @title Chuẩn bị assets, config illumination và restore checkpoint từ Drive
prepare_code = """
import json,sys
from pathlib import Path
from scripts.eval_g2 import prepare_archives,copy_atomic,sync_tree
from src.train.screen_representation import read_yaml,save_yaml,sha256_file
p=json.loads(sys.argv[1])
prepare_archives(p['archive_dir'],p['archives'],p['local_archives'],p['data'],include_test_public=True)
source=Path(p['checkpoint']); target=Path(p['weights'])
if not source.is_file(): raise FileNotFoundError(f'Missing official ViT-B/16 weights: {source}')
digest=sha256_file(source)
if not target.is_file() or sha256_file(target)!=digest: copy_atomic(source,target)
if sha256_file(target)!=digest: raise RuntimeError('Checkpoint copy SHA mismatch')
cfg=read_yaml(Path('configs/g2_experiments.yaml'))
save_yaml(cfg,Path(p['model_config']))
settings=read_yaml(Path('configs/g2_fixed_attention.yaml'))
settings.update(p['settings'],model_config=p['model_config'])
save_yaml(settings,Path(p['config']))
if p['resume']: sync_tree(p['drive_output'],p['local_output'])
print(json.dumps(dict(checkpoint_sha256=digest,settings=settings),indent=2))
"""
resolved_precision = (prior['settings']['precision'] if prior else ('bf16' if gpu['bf16_supported'] else 'fp16')) if PRECISION == 'auto' else PRECISION
settings = dict(run_id=RUN_NAME, seed=SEED, dev_seed=DEV_SEED, epochs=EPOCHS,
    batch_size=BATCH_SIZE, tile_batch_size=INFERENCE_BATCH_SIZE, num_workers=DATA_WORKERS,
    precision=resolved_precision, deterministic=DETERMINISTIC, max_dev_sources=MAX_DEV_SOURCES,
    illumination=ILLUMINATION)
if prior and prior['settings'] != settings:
    raise RuntimeError('Protocol đã đổi. Khôi phục setting cũ hoặc tạo RUN_NAME mới; không ghi đè resume.')
payload = dict(archive_dir=str(ARCHIVE_DIR),archives=ARCHIVE_NAMES,local_archives=str(ARCHIVES),
    data=str(DATA),checkpoint=str(CHECKPOINT_ON_DRIVE),weights=str(WEIGHTS),model_config=str(MODEL_CONFIG),
    config=str(CONFIG),settings=settings,resume=RESUME,local_output=str(LOCAL_OUTPUT),drive_output=str(DRIVE_OUTPUT))
subprocess.run([PYTHON,'-c',prepare_code,json.dumps(payload)],cwd=REPO,env=RUN_ENV,check=True)
environment = dict(msila_commit=MSILA_SHA,dino_commit=DINO_SHA,settings=settings,**gpu)
LOCAL_OUTPUT.mkdir(parents=True,exist_ok=True)
(LOCAL_OUTPUT/'notebook_environment.json').write_text(json.dumps(environment,indent=2)+'\n')
ENVIRONMENT_FILE.write_text(json.dumps(environment,indent=2)+'\n')

COMMON = ['--config',str(CONFIG),'--data-root',str(DATA),'--repo-dir',str(DINO),'--weights',str(WEIGHTS),
          '--output-root',str(LOCAL_OUTPUT),'--drive-output',str(DRIVE_OUTPUT),'--device','cuda']
def run_fixed(extra, label, allow_incomplete=False):
    command = [PYTHON,'scripts/run_g2_fixed.py',*COMMON,*extra]
    print('Command:', ' '.join(command))
    log_path = LOCAL_OUTPUT/f'colab_{label}.log'
    with log_path.open('a') as log:
        proc = subprocess.Popen(command,cwd=REPO,env=RUN_ENV,stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end=''); log.write(line); log.flush()
        code = proc.wait()
    shutil.copy2(log_path,DRIVE_OUTPUT/log_path.name)
    if code != 0 and not (allow_incomplete and code == 2):
        raise RuntimeError(f'{label}: exit={code}; xem failure.json/log trên Drive. Không phải PASS.')
    return code
run_fixed(['--preflight','--categories','all'], 'preflight')

## 3. Smoke — tối đa hai updates, namespace riêng

Xác minh frozen feature extraction, Adapter/Projection/Attention/Decoder gradients và native DEV Hann maps. Smoke không chạy TEST_PUBLIC và không tính vào kết quả full. CPU tests kiểm tra logic; smoke này chỉ PASS nếu GPU và assets thật chạy được.

In [ ]:
# @title Smoke GPU một category
if RUN_SMOKE:
    run_fixed(['--smoke','--categories','can',*(['--resume'] if RESUME else [])], 'smoke')

## 4. Full train E5 rồi evaluate TEST_PUBLIC bằng best.pt từng category

Huấn luyện toàn bộ TRAIN/good + native synthetic, illumination train-only, cùng seed/budget. Mỗi epoch đánh giá DEV cố định để chọn checkpoint; không train DINOv3. Sau khi hoàn tất category, runner load đúng `best.pt` để chạy toàn bộ TEST_PUBLIC của category đó. Không yêu cầu D10 lock, không tạo lock mới.

Checkpoint chứa các trainable heads, optimizer, GradScaler, RNG, cursor và config hash; frozen DINO được định danh bằng checksum. Đồng bộ Drive khi checkpoint/epoch/category kết thúc và khi có lỗi. Run hoàn thành hợp lệ được skip; không tự train lại. Không đổi setting hay selection sau khi xem TEST.

Đây là **E5 cố định**, không phải nghiệm thu đủ ablation E0–E5/D11 theo joint selection. Run All thực sự chạy training; thời gian phụ thuộc GPU, số ảnh và tile.

In [ ]:
# @title Train đủ 8 category + TEST_PUBLIC
FULL_EXIT = run_fixed(['--categories','all','--test-public',*(['--resume'] if RESUME else [])],
                      'full_train_test',allow_incomplete=True)

## 5. Kết quả thật và hình trực quan

Hiển thị synthetic DEV và real TEST_PUBLIC riêng. Macro chỉ xuất khi đủ 8/8 kết quả full hợp lệ. Không thay metric thiếu bằng 0. Không báo G2 EXP PASS (các baseline/ablation còn chưa chạy).

In [ ]:
# @title Bảng, biểu đồ và native Input / GT / Anomaly
import csv
from IPython.display import display, Markdown, Image as DisplayImage
summaries = list((LOCAL_OUTPUT/RUN_NAME).glob('*/full/summary.json'))
if len(summaries) != 1:
    raise RuntimeError(f'Cần đúng một protocol đầy đủ; tìm thấy {len(summaries)}. Không trộn run.')
SUMMARY_PATH = summaries[0]
summary = json.loads(SUMMARY_PATH.read_text())
rows = summary['categories']
table = '| Category | Training | Synthetic DEV AU-PRO@0.05 | TEST_PUBLIC | Real AU-PRO@0.05 |\n|---|---|---:|---|---:|\n'
def value(x): return '—' if x is None else f'{x:.6f}'
for r in rows:
    table += f"| {r['category']} | {r['training_status']} | {value(r['synthetic_dev_aupro_005'])} | {r['test_public_status']} | {value(r['test_public_aupro_005'])} |\n"
display(Markdown(table))
print('Fixed E5:', summary['status'],'Training',summary['training_pass'],'/8',
      'TEST_PUBLIC',summary['test_public_pass'],'/8','Macro:',summary['macro_test_public_aupro_005'])
print(summary['limitations'])
if summary['status'] == 'PASS':
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10,4))
    ax.bar([r['category'] for r in rows],[r['test_public_aupro_005'] for r in rows])
    ax.axhline(summary['macro_test_public_aupro_005'],color='red',linestyle='--',label='Macro 8/8')
    ax.set_ylabel('Real TEST_PUBLIC AU-PRO@0.05'); ax.set_ylim(0,1); ax.tick_params(axis='x',rotation=30); ax.legend()
    fig.tight_layout(); fig.savefig(SUMMARY_PATH.parent/'test_public_aupro.png',dpi=150); display(fig); plt.close(fig)
for category in CATEGORIES:
    images = sorted((SUMMARY_PATH.parent/category/'test_public/examples').glob('*/comparison.png'))
    if images:
        print(category); display(DisplayImage(filename=str(images[0])))

## 6. Export & resume

Drive lưu `best.pt`, `last.pt` cùng SHA sidecars, config/study/seed/source Git/DINO SHA/checkpoint SHA, GPU/AMP settings, loss history, DEV native maps + masks + QA, per-category TEST_PUBLIC metrics/source manifests/ví dụ, `metrics_e5_fixed.csv`, `summary.json` và logs.

Nếu Colab ngắt, mở lại notebook với cùng RUN_NAME + setting và Run All; restore Drive tự động, skip các category hoàn thành hợp lệ. Chỉ evaluate lại bằng CLI `--eval-only --categories all` khi checkpoint full đã có. Không sửa `best.pt` hoặc đổi seed/budget/illumination để resume.

In [ ]:
# @title Đồng bộ bổ sung biểu đồ và toàn bộ kết quả về Drive
sync_code = "from scripts.eval_g2 import sync_tree; import sys; sync_tree(sys.argv[1],sys.argv[2])"
subprocess.run([PYTHON,'-c',sync_code,str(LOCAL_OUTPUT),str(DRIVE_OUTPUT)],cwd=REPO,env=RUN_ENV,check=True)
print('Results:',DRIVE_OUTPUT)
print('Evaluate-only command:', ' '.join([PYTHON,'scripts/run_g2_fixed.py',*COMMON,'--eval-only','--categories','all']))
if summary['status'] != 'PASS':
    print('INCOMPLETE: xem metrics/failure.json theo category; chưa có macro nghiệm thu 8/8.')